# Chemistry Predictions with biotech-ml-toolkit

This notebook demonstrates the 6 chemistry domain models:
1. ToxicityScorer -- Tox21 12-endpoint toxicity prediction
2. SolubilityPredictor -- Aqueous solubility (log S)
3. LipophilicityPredictor -- logD and skin penetration
4. CosmeticAllergenDetector -- EU 26 fragrance allergens
5. GHSClassifier -- GHS hazard classification
6. INCISafetyScorer -- Composite INCI safety score

In [ ]:
from pathlib import Path

# All chemistry models use Morgan fingerprints from RDKit
# pip install "biotech-ml-toolkit[rdkit]"

## 1. Molecular Features

The chemistry models use Morgan fingerprints (2048-bit) and molecular descriptors from RDKit.

In [ ]:
from biotech_ml.features.molecular import smiles_to_morgan, smiles_to_descriptors

aspirin = "CC(=O)OC1=CC=CC=C1C(=O)O"

fp = smiles_to_morgan(aspirin, radius=2, n_bits=2048)
print(f"Fingerprint shape: {fp.shape}, bits on: {fp.sum()}")

desc = smiles_to_descriptors(aspirin)
for k, v in desc.items():
    print(f"  {k}: {v}")

## 2. Cosmetic Allergen Detection (no model artifacts needed)

Rule-based detection using built-in EU 26 allergen database.

In [ ]:
from biotech_ml.chemistry.allergen_detector import CosmeticAllergenDetector, EU_26_ALLERGENS

detector = CosmeticAllergenDetector()
# Load with built-in DB (no artifact files needed)
detector._allergen_db = EU_26_ALLERGENS
detector._build_index()
detector._loaded = True

result = detector.predict({
    "ingredients": [
        "AQUA", "GLYCERIN", "LINALOOL", "CITRONELLOL", 
        "GERANIOL", "COUMARIN", "BENZYL ALCOHOL"
    ]
})

print(f"Found {len(result['allergens'])} allergens:")
for a in result["allergens"]:
    print(f"  EU#{a['eu_number']}: {a['name']} (matched: {a['source_ingredient']})")

## 3. GHS Classification Constants

The GHS classifier includes a complete H-code database.

In [ ]:
from biotech_ml.chemistry.ghs_classifier import H_CODE_DESCRIPTIONS, GHSClassifier

print(f"Total H-codes: {len(H_CODE_DESCRIPTIONS)}")
for code in ["H300", "H315", "H350", "H410"]:
    print(f"  {code}: {H_CODE_DESCRIPTIONS[code]}")

# Signal word logic
print(f"\nSignal word for [H300, H315]: {GHSClassifier._determine_signal_word(['H300', 'H315'])}")
print(f"Signal word for [H315]: {GHSClassifier._determine_signal_word(['H315'])}")
print(f"Pictograms for [H300, H350]: {GHSClassifier._h_codes_to_pictograms(['H300', 'H350'])}")

## 4. Toxicity Scoring Constants

Tox21 targets and weights used in the weighted toxicity score.

In [ ]:
from biotech_ml.chemistry.toxicity_scorer import TOX21_TARGETS, TOX21_WEIGHTS

print("Tox21 Targets and Weights:")
for target in TOX21_TARGETS:
    print(f"  {target}: weight={TOX21_WEIGHTS[target]:.2f}")